In [18]:
"""
extract_rcs_ieeg.py
Extract iEEG data from RCS (Summit RC+S) .mat files only.

Output: derivatives/dbs/{subject}/{dbs_state}/{task}/{channel}.npz

Changelog:
  v1.0 - Initial release
  v1.1 - Fixed: GPi subjects get GPi-L/R filenames (not STN-L/R)
         Fixed: max_duration_sec now enforced (was defined but never checked)
"""

import numpy as np
import json
import pandas as pd
from pathlib import Path
import sys
import logging
import hashlib
from datetime import datetime
from typing import Dict, List

sys.path.append('/mnt/movement/users/jaizor/code/pd_ecog')
import imat

# =============================================================================
# CONFIGURATION
# =============================================================================

IEEG_BASE = Path('/mnt/movement/users/jaizor/xtra/data/eeg/PD_Ecog')
OUTPUT_DIR = Path('/mnt/movement/users/jaizor/xtra/derivatives/dbs')

SETTINGS = {
    'max_nan_pct':      10.0,    # Skip channels with >10% NaNs
    'min_duration_sec': 10.0,    # Skip recordings shorter than 10s
    'max_duration_sec': 3600.0,  # Skip recordings longer than 1 hour (sanity check)
    'dry_run':          False,   # Set True to test without writing files
    'verify_saved_files': True,  # Verify each .npz can be loaded after saving
    'log_level':        logging.INFO,
}

# RCS subjects
RCS_SUBJECTS = {
    'sub-01': 'sbj1_PM1_RCS',
    'sub-02': 'sbj2_PM2_RCS',           # GPi, unilateral left
    'sub-08': 'sbj8_PM8_RCS_Dystonia',  # GPi, Dystonia
    'sub-12': 'sbj12_PM12_RCS',
    'sub-13': 'sbj13_PM14_RCS',
    'sub-14': 'sbj14_PM15_RCS',
}

# Anatomical priors
ANATOMICAL_PRIORS = {
    'STN-L':       {'region': 'Subthalamic nucleus, left',            'source': 'Surgical protocol', 'uncertainty': '±5-10mm'},
    'STN-R':       {'region': 'Subthalamic nucleus, right',           'source': 'Surgical protocol', 'uncertainty': '±5-10mm'},
    'GPi-L':       {'region': 'Globus pallidus internus, left',       'source': 'Surgical protocol', 'uncertainty': '±5-10mm'},
    'GPi-R':       {'region': 'Globus pallidus internus, right',      'source': 'Surgical protocol', 'uncertainty': '±5-10mm'},
    'ECOG-8-9-L':  {'region': 'Left sensorimotor cortex (likely)',    'source': 'PMC8434942; NCT03582891', 'uncertainty': '±5-10mm'},
    'ECOG-10-11-L':{'region': 'Left sensorimotor cortex (adjacent)',  'source': 'PMC8434942; NCT03582891', 'uncertainty': '±5-10mm'},
    'ECOG-8-9-R':  {'region': 'Right sensorimotor cortex (likely)',   'source': 'PMC8434942; NCT03582891', 'uncertainty': '±5-10mm'},
    'ECOG-10-11-R':{'region': 'Right sensorimotor cortex (adjacent)', 'source': 'PMC8434942; NCT03582891', 'uncertainty': '±5-10mm'},
}

# =============================================================================
# HELPERS
# =============================================================================

def setup_logging(log_dir: Path, log_level: int = logging.INFO) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger('rcs_extraction')
    logger.setLevel(log_level)
    logger.handlers = []

    file_handler = logging.FileHandler(
        log_dir / 'rcs_extraction.log', mode='w', encoding='utf-8')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(
        logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s'))

    console_handler = logging.StreamHandler()
    console_handler.setLevel(log_level)
    console_handler.setFormatter(
        logging.Formatter('%(levelname)-8s | %(message)s'))

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    return logger


def ensure_dir(path: Path, logger: logging.Logger) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        raise RuntimeError(f"Directory creation failed: {path}")
    return path


def compute_checksum(data: np.ndarray) -> str:
    return hashlib.sha256(data.tobytes()).hexdigest()[:16]


def get_dbs_state_from_filename(filename: str) -> str:
    """Extract DBS state from RCS filename."""
    fn = filename.lower()
    if any(kw in fn for kw in ['dbson', 'dbs_on', '_on_']):
        return 'DBSON'
    elif any(kw in fn for kw in ['dbs_off', 'dbsoff', '_off_']):
        return 'DBSOFF'
    return 'UNKNOWN'


def get_task_from_filename(filename: str) -> str:
    """Extract task from RCS filename."""
    fn = filename.lower()
    if any(kw in fn for kw in ['rest', 'hand', 'foot']):
        return 'rest'
    elif any(kw in fn for kw in ['bima', 'bimanual']):
        return 'bima'
    elif any(kw in fn for kw in ['gait', 'walk']):
        return 'gait'
    return 'other'


def get_target_nucleus(sub_id: str) -> str:
    """Return 'GPi' for GPi subjects, 'STN' otherwise."""
    return 'GPi' if sub_id in ('sub-02', 'sub-08') else 'STN'


def rename_dbs_channel(raw_ch_name: str, target_nucleus: str) -> str:
    """
    FIX v1.1: imat.extract_rcs_data always names DBS channels STN-L / STN-R
    because that is what is hard-coded in imat's channel_map. For GPi subjects
    the electrode is physically in the GPi, so we rename to GPi-L / GPi-R.
    ECoG channels are left unchanged.

    Probe confirmed: sub-02 and sub-08 had STN-L/R in filename but
    target_nucleus=GPi in metadata — this caused a channel_name /
    target_nucleus mismatch in every DBS file for those subjects.
    """
    if 'ECOG' in raw_ch_name:
        return raw_ch_name  # ECoG naming is always correct
    if target_nucleus == 'GPi' and raw_ch_name.startswith('STN-'):
        return raw_ch_name.replace('STN-', 'GPi-', 1)
    return raw_ch_name


def get_channel_target(ch_name: str, subject_target: str) -> str:
    """Return the target structure for this specific channel."""
    if 'ECOG' in ch_name:
        return 'Cortex'
    return subject_target  # GPi or STN, already correct after rename


def save_channel_npz(
        data: np.ndarray,
        metadata: Dict,
        output_path: Path,
        logger: logging.Logger) -> bool:
    try:
        ensure_dir(output_path.parent, logger)
        # FIX v1.1: compute checksum on float32 (what is actually saved)
        # so the validator can reproduce it by reloading the file.
        data_f32 = data.astype(np.float32)
        metadata['data_checksum'] = compute_checksum(data_f32)

        if SETTINGS['dry_run']:
            logger.info(f"  [DRY-RUN] Would save: {output_path}")
            return True

        np.savez(output_path,
                 data=data_f32,
                 metadata=metadata,
                 allow_pickle=True)

        if SETTINGS['verify_saved_files']:
            test_load = np.load(output_path, allow_pickle=True)
            if 'data' not in test_load or test_load['data'].size == 0:
                logger.error(f"  ✗ Verification failed: {output_path.name}")
                return False

        logger.debug(f"  ✓ Saved: {output_path.name} ({output_path.stat().st_size} bytes)")
        return True

    except Exception as e:
        logger.error(f"  ✗ Save failed: {type(e).__name__}: {e}")
        return False


# =============================================================================
# EXTRACTION PIPELINE
# =============================================================================

def extract_rcs_ieeg():
    """Extract all RCS iEEG data (v1.1)."""

    logger = setup_logging(OUTPUT_DIR, SETTINGS['log_level'])

    logger.info("=" * 70)
    logger.info("RCS IEEG EXTRACTION v1.1")
    logger.info("=" * 70)
    logger.info(f"Input:  {IEEG_BASE}")
    logger.info(f"Output: {OUTPUT_DIR}")
    logger.info(f"Started: {datetime.now().isoformat()}")
    logger.info(f"Subjects: {list(RCS_SUBJECTS.keys())}")
    logger.info(
        f"Settings: max_nan_pct={SETTINGS['max_nan_pct']}%, "
        f"min_duration={SETTINGS['min_duration_sec']}s, "
        f"max_duration={SETTINGS['max_duration_sec']}s"
    )
    logger.info("=" * 70)

    ensure_dir(OUTPUT_DIR, logger)

    extraction_log: List[Dict] = []
    stats = {'total': 0, 'saved': 0, 'skipped': 0, 'failed': 0}

    for sub_idx, (sub_id, folder_name) in enumerate(RCS_SUBJECTS.items(), 1):

        folder         = IEEG_BASE / folder_name
        target_nucleus = get_target_nucleus(sub_id)
        diagnosis      = 'Dystonia' if sub_id == 'sub-08' else 'PD'

        if not folder.exists():
            logger.warning(f"⚠️  Folder not found: {folder}")
            continue

        logger.info(f"\n{'=' * 60}")
        logger.info(f"[{sub_idx}/{len(RCS_SUBJECTS)}] Subject: {sub_id}")
        logger.info(f"     Target: {target_nucleus}, Diagnosis: {diagnosis}")
        logger.info(f"{'=' * 60}")

        sub_output = ensure_dir(OUTPUT_DIR / sub_id, logger)

        subject_meta = {
            'subject_id':    sub_id,
            'folder_name':   folder_name,
            'device':        'RCS',
            'target_nucleus': target_nucleus,
            'diagnosis':     diagnosis,
            'extraction_date': datetime.now().isoformat(),
        }
        if not SETTINGS['dry_run']:
            (sub_output / 'device.txt').write_text('RCS')
            with open(sub_output / 'metadata.json', 'w') as f:
                json.dump(subject_meta, f, indent=2)

        mat_files = sorted(folder.glob('*.mat'))
        logger.info(f"  Found {len(mat_files)} .mat file(s)")

        for mat_file in mat_files:
            dbs_state = get_dbs_state_from_filename(mat_file.name)
            task_name = get_task_from_filename(mat_file.name)

            logger.info(f"\n  📄 {mat_file.name}")
            logger.info(f"     DBS: {dbs_state}, Task: {task_name}")

            rcs_data = imat.extract_rcs_data(mat_file, detrend=False)

            if not rcs_data or not rcs_data.get('channels'):
                logger.warning(f"  ⊘ No channels extracted")
                stats['failed'] += 1
                continue

            sfreq = rcs_data['sfreq']

            for raw_ch_name, ch_data in rcs_data['channels'].items():

                # FIX v1.1: rename DBS channels for GPi subjects
                ch_name   = rename_dbs_channel(raw_ch_name, target_nucleus)
                ch_target = get_channel_target(ch_name, target_nucleus)

                nan_pct = (100 * np.isnan(ch_data).sum() / ch_data.size
                           if ch_data.size > 0 else 100.0)

                if nan_pct > SETTINGS['max_nan_pct']:
                    logger.info(
                        f"    ⊘ {ch_name}: SKIP "
                        f"(NaNs={nan_pct:.1f}% > {SETTINGS['max_nan_pct']}%)")
                    stats['skipped'] += 1
                    continue

                duration_sec = len(ch_data) / sfreq

                if duration_sec > SETTINGS['max_duration_sec']:
                    logger.info(
                        f"    ⊘ {ch_name}: SKIP "
                        f"(duration={duration_sec:.1f}s > {SETTINGS['max_duration_sec']}s)")
                    stats['skipped'] += 1
                    continue

                if duration_sec < SETTINGS['min_duration_sec']:
                    logger.info(
                        f"    ⊘ {ch_name}: SKIP "
                        f"(duration={duration_sec:.1f}s < {SETTINGS['min_duration_sec']}s)")
                    stats['skipped'] += 1
                    continue

                ch_clean = (imat._interpolate_nans(ch_data)
                            if np.any(np.isnan(ch_data)) else ch_data)

                metadata = {
                    'subject_id':    sub_id,
                    'channel_name':  ch_name,
                    'target_nucleus': ch_target,
                    'hemisphere':    ('L' if ch_name.endswith('-L') else
                                      'R' if ch_name.endswith('-R') else 'Unknown'),
                    'device':        'RCS',
                    'dbs_state':     dbs_state,
                    'task':          task_name,
                    'source_file':   mat_file.name,
                    'sfreq':         float(sfreq),
                    'nan_pct':       float(nan_pct),
                    'n_samples':     int(len(ch_clean)),
                    'duration_sec':  float(duration_sec),
                    'anatomical_prior': ANATOMICAL_PRIORS.get(
                        ch_name,
                        {'region': 'Unknown',
                         'source': 'Not documented',
                         'uncertainty': 'Unverified'}),
                    'extraction_date': datetime.now().isoformat(),
                }

                output_path = sub_output / dbs_state / task_name / f'{ch_name}.npz'

                if save_channel_npz(ch_clean, metadata, output_path, logger):
                    stats['saved'] += 1
                    logger.info(
                        f"    ✓ {ch_name}: {len(ch_clean)} samples, "
                        f"{duration_sec:.1f}s, NaNs={nan_pct:.2f}%")
                    extraction_log.append({
                        'subject':        sub_id,
                        'device':         'RCS',
                        'dbs_state':      dbs_state,
                        'task':           task_name,
                        'channel':        ch_name,
                        'target_nucleus': ch_target,
                        'n_samples':      metadata['n_samples'],
                        'duration_sec':   duration_sec,
                        'nan_pct':        nan_pct,
                        'file': str(output_path.relative_to(OUTPUT_DIR)),
                    })
                else:
                    stats['failed'] += 1

                stats['total'] += 1

    # Save extraction log
    if extraction_log:
        log_df = pd.DataFrame(extraction_log)
        if not SETTINGS['dry_run']:
            log_df.to_csv(OUTPUT_DIR / 'rcs_extraction_log.csv', index=False)

        logger.info(f"\n📊 RCS Extraction Summary:")
        logger.info(
            f"  Total: {stats['total']}, Saved: {stats['saved']}, "
            f"Skipped: {stats['skipped']}, Failed: {stats['failed']}")
        logger.info(f"  Log: {OUTPUT_DIR / 'rcs_extraction_log.csv'}")
    else:
        logger.warning("⚠ No channels were extracted — check input paths and file formats")

    logger.info("\n" + "=" * 70)
    logger.info("✅ RCS EXTRACTION COMPLETE (v1.1)")
    logger.info("=" * 70)
    logger.info(f"Finished: {datetime.now().isoformat()}")
    logger.info("=" * 70)


if __name__ == '__main__':
    try:
        extract_rcs_ieeg()
    except KeyboardInterrupt:
        print("\n⚠️  Interrupted by user")
    except Exception as e:
        print(f"\n✗ CRITICAL: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

INFO     | ======================================================================
INFO     | RCS IEEG EXTRACTION v1.1
INFO     | ======================================================================
INFO     | Input:  /mnt/movement/users/jaizor/xtra/data/eeg/PD_Ecog
INFO     | Output: /mnt/movement/users/jaizor/xtra/derivatives/dbs
INFO     | Started: 2026-02-17T23:54:13.507572
INFO     | Subjects: ['sub-01', 'sub-02', 'sub-08', 'sub-12', 'sub-13', 'sub-14']
INFO     | Settings: max_nan_pct=10.0%, min_duration=10.0s, max_duration=3600.0s
INFO     | ======================================================================
INFO     | 
INFO     | [1/6] Subject: sub-01
INFO     |      Target: STN, Diagnosis: PD
INFO     | ============================================================
INFO     |   Found 4 .mat file(s)
INFO     | 
  📄 RCS_DBSOFF_bima.mat
INFO     |      DBS: DBSOFF, Task: bima
INFO     |     ✓ STN-L: 221585 samples, 886.3s, NaNs=0.07%
INFO     |     ✓ STN-R: 221585 samples, 886.

⚠️ Length mismatch detected. Truncating to 221585 samples.
⚠️ Length mismatch detected. Truncating to 208288 samples.


INFO     |     ✓ STN-L: 65468 samples, 261.9s, NaNs=0.06%
INFO     |     ✓ STN-R: 65468 samples, 261.9s, NaNs=2.52%
INFO     |     ✓ ECOG-8-9-L: 65468 samples, 261.9s, NaNs=0.06%
INFO     |     ✓ ECOG-10-11-L: 65468 samples, 261.9s, NaNs=0.06%
INFO     |     ✓ ECOG-8-9-R: 65468 samples, 261.9s, NaNs=2.52%
INFO     |     ✓ ECOG-10-11-R: 65468 samples, 261.9s, NaNs=2.52%
INFO     | 
  📄 RCS_DBSON_rest_hand_foot.mat
INFO     |      DBS: DBSON, Task: rest
INFO     |     ✓ STN-L: 226204 samples, 904.8s, NaNs=4.15%
INFO     |     ✓ STN-R: 226204 samples, 904.8s, NaNs=0.40%
INFO     |     ✓ ECOG-8-9-L: 226204 samples, 904.8s, NaNs=4.15%
INFO     |     ✓ ECOG-10-11-L: 226204 samples, 904.8s, NaNs=4.15%
INFO     |     ✓ ECOG-8-9-R: 226204 samples, 904.8s, NaNs=0.40%
INFO     |     ✓ ECOG-10-11-R: 226204 samples, 904.8s, NaNs=0.40%
INFO     | 
INFO     | [2/6] Subject: sub-02
INFO     |      Target: GPi, Diagnosis: PD
INFO     | ============================================================
INFO  

⚠️ Length mismatch detected. Truncating to 65468 samples.
⚠️ Length mismatch detected. Truncating to 226204 samples.


INFO     |     ✓ GPi-L: 193564 samples, 774.3s, NaNs=0.11%
INFO     |     ✓ ECOG-8-9-L: 193564 samples, 774.3s, NaNs=0.11%
INFO     |     ✓ ECOG-10-11-L: 193564 samples, 774.3s, NaNs=0.11%
INFO     | 
  📄 RCS_DBSOFF_gait.mat
INFO     |      DBS: DBSOFF, Task: gait
INFO     |     ✓ GPi-L: 97608 samples, 390.4s, NaNs=0.01%
INFO     |     ✓ ECOG-8-9-L: 97608 samples, 390.4s, NaNs=0.01%
INFO     |     ✓ ECOG-10-11-L: 97608 samples, 390.4s, NaNs=0.01%
INFO     | 
  📄 RCS_DBSOFF_rest_hand_foot.mat
INFO     |      DBS: DBSOFF, Task: rest
INFO     |     ⊘ GPi-L: SKIP (NaNs=24.0% > 10.0%)
INFO     |     ⊘ ECOG-8-9-L: SKIP (NaNs=24.0% > 10.0%)
INFO     |     ⊘ ECOG-10-11-L: SKIP (NaNs=24.0% > 10.0%)
INFO     | 
  📄 RCS_DBSON_gait.mat
INFO     |      DBS: DBSON, Task: gait
INFO     |     ✓ GPi-L: 77859 samples, 311.4s, NaNs=0.59%
INFO     |     ✓ ECOG-8-9-L: 77859 samples, 311.4s, NaNs=0.59%
INFO     |     ✓ ECOG-10-11-L: 77859 samples, 311.4s, NaNs=0.59%
INFO     | 
  📄 RCS_DBSON_rest_hand_foot.

...

⚠️ Length mismatch detected. Truncating to 189950 samples.


INFO     |     ✓ STN-L: 438257 samples, 1753.0s, NaNs=0.04%
INFO     |     ✓ STN-R: 438257 samples, 1753.0s, NaNs=0.12%
INFO     |     ✓ ECOG-8-9-L: 438257 samples, 1753.0s, NaNs=0.04%
INFO     |     ✓ ECOG-10-11-L: 438257 samples, 1753.0s, NaNs=0.04%
INFO     |     ✓ ECOG-8-9-R: 438257 samples, 1753.0s, NaNs=0.12%
INFO     |     ✓ ECOG-10-11-R: 438257 samples, 1753.0s, NaNs=0.12%
INFO     | 
  📄 RCS_DBSOFF_rest_hand_foot.mat
INFO     |      DBS: DBSOFF, Task: rest
INFO     |     ✓ STN-L: 96479 samples, 385.9s, NaNs=0.06%
INFO     |     ✓ STN-R: 96479 samples, 385.9s, NaNs=0.38%
INFO     |     ✓ ECOG-8-9-L: 96479 samples, 385.9s, NaNs=0.06%
INFO     |     ✓ ECOG-10-11-L: 96479 samples, 385.9s, NaNs=0.06%
INFO     |     ✓ ECOG-8-9-R: 96479 samples, 385.9s, NaNs=0.38%
INFO     |     ✓ ECOG-10-11-R: 96479 samples, 385.9s, NaNs=0.38%
INFO     | 
  📄 RCS_DBSON_gait.mat
INFO     |      DBS: DBSON, Task: gait
INFO     |     ✓ STN-L: 106050 samples, 424.2s, NaNs=0.00%
INFO     |     ✓ STN-R: 1

⚠️ Length mismatch detected. Truncating to 438257 samples.
⚠️ Length mismatch detected. Truncating to 96479 samples.
⚠️ Length mismatch detected. Truncating to 106050 samples.


INFO     |     ✓ ECOG-10-11-R: 106050 samples, 424.2s, NaNs=0.42%
INFO     | 
  📄 RCS_DBSON_rest_hand_foot.mat
INFO     |      DBS: DBSON, Task: rest
INFO     |     ✓ STN-L: 279245 samples, 1117.0s, NaNs=0.02%
INFO     |     ✓ STN-R: 279245 samples, 1117.0s, NaNs=0.06%
INFO     |     ✓ ECOG-8-9-L: 279245 samples, 1117.0s, NaNs=0.02%
INFO     |     ✓ ECOG-10-11-L: 279245 samples, 1117.0s, NaNs=0.02%
INFO     |     ✓ ECOG-8-9-R: 279245 samples, 1117.0s, NaNs=0.06%
INFO     |     ✓ ECOG-10-11-R: 279245 samples, 1117.0s, NaNs=0.06%
INFO     | 
📊 RCS Extraction Summary:
INFO     |   Total: 138, Saved: 138, Skipped: 3, Failed: 0
INFO     |   Log: /mnt/movement/users/jaizor/xtra/derivatives/dbs/rcs_extraction_log.csv
INFO     | 
INFO     | ✅ RCS EXTRACTION COMPLETE (v1.1)
INFO     | ======================================================================
INFO     | Finished: 2026-02-17T23:54:17.669051
INFO     | ======================================================================


⚠️ Length mismatch detected. Truncating to 279245 samples.


In [19]:
"""
extract_percept_ieeg.py
Extract iEEG data from Percept .mat files only.

Key difference from RCS: DBS state is in segment Notes, not filename.
Output: derivatives/dbs/{subject}/{dbs_state}/{task}/{channel}.npz

Changelog:
  v1.0 - Initial release
  v1.1 - Fixed: Mixed DBS state parsing ("Gait ON then OFF" → flag as mixed_state)
         Fixed: GPi-aware channel naming (GPi-L not STN-L for GPi subjects)
         Fixed: max_duration_sec validation now enforced
         Fixed: Checksum computed on float32 (what is saved), not float64 input
         Aligned: NaN threshold docs with extraction settings (10%)
"""

import numpy as np
import json
import pandas as pd
from pathlib import Path
import sys
import logging
import hashlib
import re
from datetime import datetime
from typing import Dict, List, Optional

sys.path.append('/mnt/movement/users/jaizor/code/pd_ecog')
import imat

# =============================================================================
# CONFIGURATION
# =============================================================================

IEEG_BASE = Path('/mnt/movement/users/jaizor/xtra/data/eeg/PD_Ecog')
OUTPUT_DIR = Path('/mnt/movement/users/jaizor/xtra/derivatives/dbs')

SETTINGS = {
    'max_nan_pct':      10.0,    # Skip channels with >10% NaNs
    'min_duration_sec': 10.0,    # Skip recordings shorter than 10s
    'max_duration_sec': 3600.0,  # Skip recordings longer than 1 hour (sanity check)
    'dry_run':          False,
    'verify_saved_files': True,
    'log_level':        logging.INFO,
}

PERCEPT_SUBJECTS = {
    'sub-03': 'sbj3_PM3_Percept',      # GPi
    'sub-04': 'sbj4_PM4_Percept',
    'sub-05': 'sbj5_PM5_Percept',
    'sub-06': 'sbj6_PM6_Percept',
    'sub-07': 'sbj7_PM7_Percept',
    'sub-09': 'sbj9_PM9_Percept',      # GPi
    'sub-10': 'sbj10_PM10_Percept',
    'sub-11': 'sbj11_PM11_Percept',    # Split Left/Right files
}

ANATOMICAL_PRIORS = {
    'STN-L': {'region': 'Subthalamic nucleus, left',       'source': 'Surgical protocol', 'uncertainty': '±5-10mm'},
    'STN-R': {'region': 'Subthalamic nucleus, right',      'source': 'Surgical protocol', 'uncertainty': '±5-10mm'},
    'GPi-L': {'region': 'Globus pallidus internus, left',  'source': 'Surgical protocol', 'uncertainty': '±5-10mm'},
    'GPi-R': {'region': 'Globus pallidus internus, right', 'source': 'Surgical protocol', 'uncertainty': '±5-10mm'},
}

# =============================================================================
# HELPERS
# =============================================================================

def setup_logging(log_dir: Path, log_level: int = logging.INFO) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger('percept_extraction')
    logger.setLevel(log_level)
    logger.handlers = []

    file_handler = logging.FileHandler(
        log_dir / 'percept_extraction.log', mode='w', encoding='utf-8')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(
        logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s'))

    console_handler = logging.StreamHandler()
    console_handler.setLevel(log_level)
    console_handler.setFormatter(
        logging.Formatter('%(levelname)-8s | %(message)s'))

    logger.addHandler(file_handler)
    logger.addHandler(console_handler)
    return logger


def ensure_dir(path: Path, logger: logging.Logger) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        raise RuntimeError(f"Directory creation failed: {path}")
    return path


def compute_checksum(data: np.ndarray) -> str:
    return hashlib.sha256(data.tobytes()).hexdigest()[:16]


def get_dbs_state_from_notes(notes: str) -> tuple:
    """
    Extract DBS state from Percept segment Notes field.

    Returns (dbs_state, mixed_state).
    dbs_state : 'DBSON', 'DBSOFF', or 'UNKNOWN'
    mixed_state : True if recording contains both ON and OFF states.
    """
    notes_lower = notes.lower()
    has_on  = bool(re.search(r'\bon\b',  notes_lower))
    has_off = bool(re.search(r'\boff\b', notes_lower))

    if has_on and has_off:
        on_match  = re.search(r'\bon\b',  notes_lower)
        off_match = re.search(r'\boff\b', notes_lower)
        if on_match and off_match:
            if on_match.start() < off_match.start():
                return 'DBSON',  True
            else:
                return 'DBSOFF', True
        return 'UNKNOWN', True

    if has_on:
        return 'DBSON',  False
    if has_off:
        return 'DBSOFF', False
    return 'UNKNOWN', False


def get_task_from_notes(notes: str) -> str:
    notes_lower = notes.lower()
    if any(kw in notes_lower for kw in ['gait', 'walk']):
        return 'gait'
    if any(kw in notes_lower for kw in ['bima', 'bimanual']):
        return 'bima'
    if any(kw in notes_lower for kw in ['rest', 'rhf']):
        return 'rest'
    return 'other'


def get_target_nucleus(sub_id: str) -> str:
    return 'GPi' if sub_id in ('sub-03', 'sub-09') else 'STN'


def get_channel_name(cond_info: Dict, target_nucleus: str) -> str:
    """Channel label contains 'LEFT' or 'RIGHT' — map to nucleus-hemisphere name."""
    channel_label = cond_info.get('channel', '').upper()
    if 'LEFT' in channel_label:
        return f'{target_nucleus}-L'
    if 'RIGHT' in channel_label:
        return f'{target_nucleus}-R'
    return cond_info.get('channel', 'UNKNOWN')


def save_channel_npz(
        data: np.ndarray,
        metadata: Dict,
        output_path: Path,
        logger: logging.Logger) -> bool:
    try:
        ensure_dir(output_path.parent, logger)

        # FIX v1.1: compute checksum on float32 (what is actually saved to disk)
        # so the validator can reproduce it by simply reloading the file.
        data_f32 = data.astype(np.float32)
        metadata['data_checksum'] = compute_checksum(data_f32)

        if SETTINGS['dry_run']:
            logger.info(f"  [DRY-RUN] Would save: {output_path}")
            return True

        np.savez(output_path,
                 data=data_f32,
                 metadata=metadata,
                 allow_pickle=True)

        if SETTINGS['verify_saved_files']:
            test_load = np.load(output_path, allow_pickle=True)
            if 'data' not in test_load or test_load['data'].size == 0:
                logger.error(f"  ✗ Verification failed: {output_path.name}")
                return False

        logger.debug(f"  ✓ Saved: {output_path.name} ({output_path.stat().st_size} bytes)")
        return True

    except Exception as e:
        logger.error(f"  ✗ Save failed: {type(e).__name__}: {e}")
        return False


# =============================================================================
# EXTRACTION PIPELINE
# =============================================================================

def extract_percept_ieeg():
    """Extract all Percept iEEG data (v1.1)."""

    logger = setup_logging(OUTPUT_DIR, SETTINGS['log_level'])

    logger.info("=" * 70)
    logger.info("PERCEPT IEEG EXTRACTION v1.1")
    logger.info("=" * 70)
    logger.info(f"Input:  {IEEG_BASE}")
    logger.info(f"Output: {OUTPUT_DIR}")
    logger.info(f"Started: {datetime.now().isoformat()}")
    logger.info(f"Subjects: {list(PERCEPT_SUBJECTS.keys())}")
    logger.info(
        f"Settings: max_nan_pct={SETTINGS['max_nan_pct']}%, "
        f"min_duration={SETTINGS['min_duration_sec']}s, "
        f"max_duration={SETTINGS['max_duration_sec']}s"
    )
    logger.info("=" * 70)

    ensure_dir(OUTPUT_DIR, logger)

    extraction_log: List[Dict] = []
    stats = {'total': 0, 'saved': 0, 'skipped': 0, 'failed': 0}

    for sub_idx, (sub_id, folder_name) in enumerate(PERCEPT_SUBJECTS.items(), 1):

        folder         = IEEG_BASE / folder_name
        target_nucleus = get_target_nucleus(sub_id)

        if not folder.exists():
            logger.warning(f"⚠️  Folder not found: {folder}")
            continue

        logger.info(f"\n{'=' * 60}")
        logger.info(f"[{sub_idx}/{len(PERCEPT_SUBJECTS)}] Subject: {sub_id}")
        logger.info(f"     Target: {target_nucleus}, Diagnosis: PD")
        logger.info(f"{'=' * 60}")

        sub_output = ensure_dir(OUTPUT_DIR / sub_id, logger)

        subject_meta = {
            'subject_id':    sub_id,
            'folder_name':   folder_name,
            'device':        'Percept',
            'target_nucleus': target_nucleus,
            'diagnosis':     'PD',
            'extraction_date': datetime.now().isoformat(),
        }
        if not SETTINGS['dry_run']:
            (sub_output / 'device.txt').write_text('Percept')
            with open(sub_output / 'metadata.json', 'w') as f:
                json.dump(subject_meta, f, indent=2)

        mat_files = sorted(folder.glob('*.mat'))
        logger.info(f"  Found {len(mat_files)} .mat file(s)")

        for mat_file in mat_files:
            logger.info(f"\n  📄 {mat_file.name}")

            percept_data = imat.extract_percept_data(mat_file, condition_keyword=None)

            if not percept_data:
                logger.warning(f"  ⊘ No conditions extracted")
                stats['failed'] += 1
                continue

            for cond_name, cond_info in percept_data.items():
                dbs_state, mixed_state = get_dbs_state_from_notes(cond_name)
                task_name = get_task_from_notes(cond_name)
                ch_name   = get_channel_name(cond_info, target_nucleus)

                logger.info(f"    Condition: {cond_name}")
                logger.info(
                    f"    → DBS: {dbs_state}"
                    f"{' (mixed)' if mixed_state else ''}, Task: {task_name}")

                ch_data = cond_info['data']
                sfreq   = cond_info['sfreq']

                nan_pct = (100 * np.isnan(ch_data).sum() / ch_data.size
                           if ch_data.size > 0 else 100.0)

                if nan_pct > SETTINGS['max_nan_pct']:
                    logger.info(
                        f"      ⊘ {ch_name}: SKIP "
                        f"(NaNs={nan_pct:.1f}% > {SETTINGS['max_nan_pct']}%)")
                    stats['skipped'] += 1
                    continue

                duration_sec = len(ch_data) / sfreq

                if duration_sec > SETTINGS['max_duration_sec']:
                    logger.info(
                        f"      ⊘ {ch_name}: SKIP "
                        f"(duration={duration_sec:.1f}s > {SETTINGS['max_duration_sec']}s)")
                    stats['skipped'] += 1
                    continue

                if duration_sec < SETTINGS['min_duration_sec']:
                    logger.info(
                        f"      ⊘ {ch_name}: SKIP "
                        f"(duration={duration_sec:.1f}s < {SETTINGS['min_duration_sec']}s)")
                    stats['skipped'] += 1
                    continue

                ch_clean = (imat._interpolate_nans(ch_data)
                            if np.any(np.isnan(ch_data)) else ch_data)

                metadata = {
                    'subject_id':    sub_id,
                    'channel_name':  ch_name,
                    'target_nucleus': target_nucleus,
                    'hemisphere':    ('L' if ch_name.endswith('-L') else
                                      'R' if ch_name.endswith('-R') else 'Unknown'),
                    'device':        'Percept',
                    'dbs_state':     dbs_state,
                    'mixed_state':   mixed_state,
                    'task':          task_name,
                    'source_file':   mat_file.name,
                    'condition_name': cond_name,
                    'sfreq':         float(sfreq),
                    'nan_pct':       float(nan_pct),
                    'n_samples':     int(len(ch_clean)),
                    'duration_sec':  float(duration_sec),
                    'anatomical_prior': ANATOMICAL_PRIORS.get(
                        ch_name,
                        {'region': 'Unknown',
                         'source': 'Not documented',
                         'uncertainty': 'Unverified'}),
                    'extraction_date': datetime.now().isoformat(),
                }

                output_path = (sub_output / dbs_state / task_name
                               / f'{ch_name}_{cond_name.replace(" ", "_")}.npz')

                if save_channel_npz(ch_clean, metadata, output_path, logger):
                    stats['saved'] += 1
                    logger.info(
                        f"      ✓ {ch_name}: {len(ch_clean)} samples, "
                        f"{duration_sec:.1f}s, NaNs={nan_pct:.2f}%")
                    extraction_log.append({
                        'subject':        sub_id,
                        'device':         'Percept',
                        'dbs_state':      dbs_state,
                        'mixed_state':    mixed_state,
                        'task':           task_name,
                        'channel':        ch_name,
                        'target_nucleus': target_nucleus,
                        'n_samples':      metadata['n_samples'],
                        'duration_sec':   duration_sec,
                        'nan_pct':        nan_pct,
                        'file': str(output_path.relative_to(OUTPUT_DIR)),
                    })
                else:
                    stats['failed'] += 1

                stats['total'] += 1

    if extraction_log:
        log_df = pd.DataFrame(extraction_log)
        if not SETTINGS['dry_run']:
            log_df.to_csv(OUTPUT_DIR / 'percept_extraction_log.csv', index=False)

        mixed_count = log_df['mixed_state'].sum() if 'mixed_state' in log_df.columns else 0

        logger.info(f"\n📊 Percept Extraction Summary:")
        logger.info(
            f"  Total: {stats['total']}, Saved: {stats['saved']}, "
            f"Skipped: {stats['skipped']}, Failed: {stats['failed']}")
        logger.info(f"  Mixed-state recordings: {mixed_count} (flagged for separate analysis)")
        logger.info(f"  Log: {OUTPUT_DIR / 'percept_extraction_log.csv'}")
    else:
        logger.warning("⚠ No channels extracted — check input paths and file formats")

    logger.info("\n" + "=" * 70)
    logger.info("✅ PERCEPT EXTRACTION COMPLETE (v1.1)")
    logger.info("=" * 70)
    logger.info(f"Finished: {datetime.now().isoformat()}")
    logger.info("=" * 70)


if __name__ == '__main__':
    try:
        extract_percept_ieeg()
    except KeyboardInterrupt:
        print("\n⚠️  Interrupted by user")
    except Exception as e:
        print(f"\n✗ CRITICAL: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

INFO     | ======================================================================
INFO     | PERCEPT IEEG EXTRACTION v1.1
INFO     | ======================================================================
INFO     | Input:  /mnt/movement/users/jaizor/xtra/data/eeg/PD_Ecog
INFO     | Output: /mnt/movement/users/jaizor/xtra/derivatives/dbs
INFO     | Started: 2026-02-17T23:54:17.806738
INFO     | Subjects: ['sub-03', 'sub-04', 'sub-05', 'sub-06', 'sub-07', 'sub-09', 'sub-10', 'sub-11']
INFO     | Settings: max_nan_pct=10.0%, min_duration=10.0s, max_duration=3600.0s
INFO     | ======================================================================
INFO     | 
INFO     | [1/8] Subject: sub-03
INFO     |      Target: GPi, Diagnosis: PD
INFO     | ============================================================
INFO     |   Found 1 .mat file(s)
INFO     | 
  📄 Percept_JSON_parsed.mat
INFO     |     Condition: RHF OFF Left
INFO     |     → DBS: DBSOFF, Task: rest
INFO     |       ✓ GPi-L: 158188 sa

In [20]:
"""
validate_extraction.py
----------------------
Independent post-extraction validator. Reads saved .npz files and raw .mat
sources to verify the extraction was correct. Run AFTER both extraction
scripts complete.

Checks performed:
  1. Every .npz loads without error and contains 'data' and 'metadata'
  2. Data array is non-empty and finite (no remaining NaNs/Infs)
  3. Checksum in metadata matches recomputed checksum of stored data
  4. Metadata fields are internally consistent (channel_name ↔ target_nucleus,
     hemisphere ↔ channel suffix, device matches folder device.txt)
  5. Duration in metadata matches actual array length / sfreq
  6. No subject is entirely missing from output
  7. Mixed-state files are flagged in metadata
  8. RCS GPi subjects: channel name prefix matches target nucleus
  9. Cross-check: every condition seen in raw .mat appears in output
     (catches silent skips that shouldn't have happened)

Output: validation_report.txt and validation_report.csv in OUTPUT_DIR
"""

import numpy as np
import json
import hashlib
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime
from scipy.io import loadmat

sys.path.append('/mnt/movement/users/jaizor/code/pd_ecog')
import imat

OUTPUT_DIR = Path('/mnt/movement/users/jaizor/xtra/derivatives/dbs')
IEEG_BASE  = Path('/mnt/movement/users/jaizor/xtra/data/eeg/PD_Ecog')

ALL_SUBJECTS = {
    # RCS
    'sub-01': ('sbj1_PM1_RCS',          'RCS'),
    'sub-02': ('sbj2_PM2_RCS',          'RCS'),
    'sub-08': ('sbj8_PM8_RCS_Dystonia', 'RCS'),
    'sub-12': ('sbj12_PM12_RCS',        'RCS'),
    'sub-13': ('sbj13_PM14_RCS',        'RCS'),
    'sub-14': ('sbj14_PM15_RCS',        'RCS'),
    # Percept
    'sub-03': ('sbj3_PM3_Percept',  'Percept'),
    'sub-04': ('sbj4_PM4_Percept',  'Percept'),
    'sub-05': ('sbj5_PM5_Percept',  'Percept'),
    'sub-06': ('sbj6_PM6_Percept',  'Percept'),
    'sub-07': ('sbj7_PM7_Percept',  'Percept'),
    'sub-09': ('sbj9_PM9_Percept',  'Percept'),
    'sub-10': ('sbj10_PM10_Percept','Percept'),
    'sub-11': ('sbj11_PM11_Percept','Percept'),
}

GPi_SUBJECTS = {
    'sub-02': 'GPi',  # RCS
    'sub-08': 'GPi',  # RCS
    'sub-03': 'GPi',  # Percept
    'sub-09': 'GPi',  # Percept
}

SETTINGS = {
    'max_nan_pct':     10.0,
    'min_duration_sec': 10.0,
    'max_duration_sec': 3600.0,
}

SEP = "=" * 70

# =============================================================================
# HELPERS
# =============================================================================

def recompute_checksum(data: np.ndarray) -> str:
    return hashlib.sha256(data.tobytes()).hexdigest()[:16]


def load_npz(path: Path):
    """Load .npz, return (data, meta, error_str)."""
    try:
        npz  = np.load(path, allow_pickle=True)
        data = npz['data']
        meta = npz['metadata'].item()
        if not isinstance(meta, dict):
            return None, None, "metadata is not a dict"
        return data, meta, None
    except Exception as e:
        return None, None, str(e)


def check_file(path: Path, sub_id: str, device: str) -> list:
    """Run all per-file checks. Returns list of issue strings (empty = OK)."""
    issues = []
    data, meta, err = load_npz(path)

    # CHECK 1: loads correctly
    if err:
        return [f"LOAD ERROR: {err}"]

    # CHECK 2: data is non-empty and finite
    if data is None or data.size == 0:
        issues.append("data array is empty")
    else:
        if np.any(np.isnan(data)):
            issues.append(f"data contains NaNs ({np.isnan(data).sum()} samples)")
        if np.any(np.isinf(data)):
            issues.append(f"data contains Infs")

    # CHECK 3: checksum matches.
    # Extraction scripts v1.1+ compute the checksum on float32 (after astype cast),
    # which is the same dtype stored in and loaded from the .npz file.
    # So we recompute directly on the loaded array — no cast needed.
    stored_cs = meta.get('data_checksum', None)
    if stored_cs is None:
        issues.append("data_checksum key missing from metadata")
    elif data is not None:
        actual_cs = recompute_checksum(data)  # data is float32 as loaded
        if stored_cs != actual_cs:
            issues.append(f"checksum mismatch: stored={stored_cs}, actual={actual_cs}")

    # CHECK 4a: required metadata keys present
    required_keys = ['subject_id', 'channel_name', 'target_nucleus', 'hemisphere',
                     'device', 'dbs_state', 'task', 'sfreq', 'nan_pct',
                     'n_samples', 'duration_sec', 'source_file']
    for k in required_keys:
        if k not in meta:
            issues.append(f"metadata missing key: '{k}'")

    if issues:  # stop here if keys missing
        return issues

    # CHECK 4b: subject_id matches folder
    if meta['subject_id'] != sub_id:
        issues.append(f"subject_id mismatch: stored={meta['subject_id']}, expected={sub_id}")

    # CHECK 4c: device matches device.txt
    device_file = OUTPUT_DIR / sub_id / 'device.txt'
    if device_file.exists():
        stored_device = device_file.read_text().strip()
        if meta['device'] != stored_device:
            issues.append(f"device mismatch: metadata={meta['device']}, device.txt={stored_device}")

    # CHECK 4d: channel_name prefix matches target_nucleus
    ch_name   = meta['channel_name']
    ch_target = meta['target_nucleus']
    if ch_target not in ('Cortex',):  # Cortex channels are ECOG-*, that's fine
        ch_prefix = ch_name.split('-')[0] if '-' in ch_name else ''
        if ch_prefix != ch_target:
            issues.append(
                f"channel_name/target_nucleus mismatch: "
                f"channel={ch_name}, target={ch_target} "
                f"(prefix '{ch_prefix}' ≠ '{ch_target}')"
            )

    # CHECK 4e: hemisphere suffix matches channel name
    ch_name   = meta['channel_name']
    hemi      = meta['hemisphere']
    expected_hemi = 'L' if ch_name.endswith('-L') else ('R' if ch_name.endswith('-R') else 'Unknown')
    if hemi != expected_hemi:
        issues.append(f"hemisphere mismatch: stored={hemi}, expected={expected_hemi} (from {ch_name})")

    # CHECK 4f: dbs_state is a known value
    if meta['dbs_state'] not in ('DBSON', 'DBSOFF', 'UNKNOWN'):
        issues.append(f"unexpected dbs_state: {meta['dbs_state']}")

    # CHECK 4g: task is a known value
    if meta['task'] not in ('rest', 'bima', 'gait', 'other'):
        issues.append(f"unexpected task: {meta['task']}")

    # CHECK 5: duration matches actual array
    if data is not None:
        sfreq        = meta.get('sfreq', 0)
        stored_dur   = meta.get('duration_sec', 0)
        stored_nsamp = meta.get('n_samples', 0)
        actual_nsamp = len(data)
        if actual_nsamp != stored_nsamp:
            issues.append(f"n_samples mismatch: stored={stored_nsamp}, actual={actual_nsamp}")
        if sfreq > 0:
            actual_dur = actual_nsamp / sfreq
            if abs(actual_dur - stored_dur) > 0.1:  # 0.1s tolerance
                issues.append(f"duration mismatch: stored={stored_dur:.2f}s, actual={actual_dur:.2f}s")

    # CHECK 6 (data quality bounds): nan_pct should be within settings
    nan_pct = meta.get('nan_pct', 0)
    if nan_pct > SETTINGS['max_nan_pct']:
        issues.append(f"nan_pct={nan_pct:.1f}% exceeds threshold {SETTINGS['max_nan_pct']}% (should have been skipped)")

    dur = meta.get('duration_sec', 0)
    if dur < SETTINGS['min_duration_sec']:
        issues.append(f"duration={dur:.1f}s below min {SETTINGS['min_duration_sec']}s (should have been skipped)")
    if dur > SETTINGS['max_duration_sec']:
        issues.append(f"duration={dur:.1f}s exceeds max {SETTINGS['max_duration_sec']}s (should have been skipped)")

    # CHECK 7: mixed_state flag present for Percept files (device='Percept')
    if meta.get('device') == 'Percept' and 'mixed_state' not in meta:
        issues.append("Percept file missing 'mixed_state' key (old extraction run?)")

    return issues


# =============================================================================
# CHECK 9: Cross-check raw .mat → every condition should appear in output
# =============================================================================

def check_percept_coverage(sub_id: str, folder_name: str) -> list:
    """
    Read raw Percept .mat files and verify every condition that passes
    quality filters appears in the output directory.
    Returns list of missing condition strings.
    """
    missing = []
    folder = IEEG_BASE / folder_name
    if not folder.exists():
        return [f"raw folder not found: {folder}"]

    sub_out = OUTPUT_DIR / sub_id

    for mat_file in sorted(folder.glob('*.mat')):
        try:
            percept_data = imat.extract_percept_data(mat_file, condition_keyword=None)
        except Exception as e:
            missing.append(f"ERROR reading {mat_file.name}: {e}")
            continue

        for cond_name, cond_info in percept_data.items():
            ch_data  = cond_info['data']
            sfreq    = cond_info['sfreq']
            nan_pct  = 100 * np.isnan(ch_data).sum() / ch_data.size if ch_data.size > 0 else 100
            duration = len(ch_data) / sfreq

            # Skip conditions that should legitimately be filtered
            if nan_pct > SETTINGS['max_nan_pct']:
                continue
            if duration < SETTINGS['min_duration_sec']:
                continue
            if duration > SETTINGS['max_duration_sec']:
                continue

            # Check that at least one .npz mentions this condition_name
            cond_key = cond_name.replace(' ', '_')
            matches  = list(sub_out.rglob(f'*{cond_key}*.npz'))
            if not matches:
                missing.append(f"{mat_file.name} → condition '{cond_name}' not found in output")

    return missing


def check_rcs_coverage(sub_id: str, folder_name: str) -> list:
    """
    Read raw RCS .mat files and verify every channel×file that passes
    quality filters appears in output.

    FIX: imat.extract_rcs_data always returns DBS channels as STN-L/STN-R
    (hardcoded in imat channel_map). For GPi subjects (sub-02, sub-08) the
    extraction script renames these to GPi-L/GPi-R before saving. We apply
    the same rename here before searching for output files — otherwise we get
    false-positive coverage warnings for those subjects.
    """
    GPi_RCS = ('sub-02', 'sub-08')
    missing = []
    folder = IEEG_BASE / folder_name
    if not folder.exists():
        return [f"raw folder not found: {folder}"]

    sub_out = OUTPUT_DIR / sub_id

    for mat_file in sorted(folder.glob('*.mat')):
        try:
            rcs_data = imat.extract_rcs_data(mat_file, detrend=False)
        except Exception as e:
            missing.append(f"ERROR reading {mat_file.name}: {e}")
            continue

        if not rcs_data or not rcs_data.get('channels'):
            continue

        sfreq = rcs_data['sfreq']

        for raw_ch_name, ch_data in rcs_data['channels'].items():
            nan_pct  = 100 * np.isnan(ch_data).sum() / ch_data.size if ch_data.size > 0 else 100
            duration = len(ch_data) / sfreq

            if nan_pct > SETTINGS['max_nan_pct']:
                continue
            if duration < SETTINGS['min_duration_sec']:
                continue
            if duration > SETTINGS['max_duration_sec']:
                continue

            # Mirror the rename_dbs_channel logic from extract_rcs_ieeg.py v1.1
            if sub_id in GPi_RCS and 'ECOG' not in raw_ch_name and raw_ch_name.startswith('STN-'):
                ch_name = raw_ch_name.replace('STN-', 'GPi-', 1)
            else:
                ch_name = raw_ch_name

            matches = list(sub_out.rglob(f'{ch_name}.npz'))
            if not matches:
                missing.append(f"{mat_file.name} → channel '{ch_name}' not found in output")

    return missing


# =============================================================================
# MAIN VALIDATION LOOP
# =============================================================================

print(f"\n{SEP}")
print("  POST-EXTRACTION VALIDATOR")
print(f"  Started: {datetime.now().isoformat()}")
print(SEP)

all_results = []   # for CSV
report_lines = []  # for .txt

total_files   = 0
total_issues  = 0
total_ok      = 0

for sub_id, (folder_name, device) in ALL_SUBJECTS.items():
    sub_dir = OUTPUT_DIR / sub_id
    print(f"\n{'='*50}")
    print(f"  {sub_id} [{device}]")
    print('='*50)

    # CHECK 0: subject output dir exists
    if not sub_dir.exists():
        msg = f"  ❌ Output directory missing: {sub_dir}"
        print(msg)
        report_lines.append(msg)
        all_results.append({'subject': sub_id, 'device': device, 'file': 'N/A',
                             'status': 'MISSING_DIR', 'issues': 'Output dir not found'})
        continue

    npz_files = list(sub_dir.rglob('*.npz'))
    if not npz_files:
        msg = f"  ❌ No .npz files found in {sub_dir}"
        print(msg)
        report_lines.append(msg)
        all_results.append({'subject': sub_id, 'device': device, 'file': 'N/A',
                             'status': 'NO_FILES', 'issues': 'No .npz files'})
        continue

    print(f"  Found {len(npz_files)} .npz files")

    # Per-file checks (checks 1–8)
    sub_issues = 0
    for f in sorted(npz_files):
        issues = check_file(f, sub_id, device)
        total_files += 1
        rel = f.relative_to(OUTPUT_DIR)
        if issues:
            total_issues += len(issues)
            sub_issues   += len(issues)
            for iss in issues:
                msg = f"  ❌ {rel}: {iss}"
                print(msg)
                report_lines.append(msg)
            all_results.append({'subject': sub_id, 'device': device,
                                 'file': str(rel), 'status': 'FAIL',
                                 'issues': ' | '.join(issues)})
        else:
            total_ok += 1
            all_results.append({'subject': sub_id, 'device': device,
                                 'file': str(rel), 'status': 'OK', 'issues': ''})

    if sub_issues == 0:
        print(f"  ✅ All {len(npz_files)} files pass per-file checks")
    else:
        print(f"  ⚠ {sub_issues} issue(s) across {len(npz_files)} files")

    # CHECK 9: coverage cross-check against raw .mat
    print(f"  Cross-checking coverage against raw .mat files...")
    if device == 'Percept':
        missing = check_percept_coverage(sub_id, folder_name)
    else:
        missing = check_rcs_coverage(sub_id, folder_name)

    if missing:
        for m in missing:
            msg = f"  ⚠ COVERAGE: {m}"
            print(msg)
            report_lines.append(msg)
            all_results.append({'subject': sub_id, 'device': device,
                                 'file': 'coverage_check', 'status': 'MISSING',
                                 'issues': m})
    else:
        print(f"  ✅ Coverage OK — all expected conditions found in output")

# =============================================================================
# SUMMARY
# =============================================================================

print(f"\n{SEP}")
print("  VALIDATION SUMMARY")
print(SEP)
print(f"  Files checked:  {total_files}")
print(f"  Files OK:       {total_ok}")
print(f"  Issues found:   {total_issues}")

if total_issues == 0 and all(r['status'] in ('OK',) for r in all_results
                              if r['file'] not in ('N/A', 'coverage_check')):
    print("\n  ✅ ALL CHECKS PASSED — extraction looks correct")
else:
    print("\n  ❌ ISSUES FOUND — review report before proceeding")

print(f"\n  Finished: {datetime.now().isoformat()}")
print(SEP)

# Save CSV
df = pd.DataFrame(all_results)
csv_path = OUTPUT_DIR / 'validation_report.csv'
df.to_csv(csv_path, index=False)
print(f"\n  Report saved: {csv_path}")

# Save text report
txt_path = OUTPUT_DIR / 'validation_report.txt'
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))
print(f"  Issues log:   {txt_path}")


  POST-EXTRACTION VALIDATOR
  Started: 2026-02-17T23:54:18.621038

  sub-01 [RCS]
  Found 24 .npz files
  ✅ All 24 files pass per-file checks
  Cross-checking coverage against raw .mat files...
⚠️ Length mismatch detected. Truncating to 221585 samples.
⚠️ Length mismatch detected. Truncating to 208288 samples.
⚠️ Length mismatch detected. Truncating to 65468 samples.
⚠️ Length mismatch detected. Truncating to 226204 samples.
  ✅ Coverage OK — all expected conditions found in output

  sub-02 [RCS]
  Found 12 .npz files
  ✅ All 12 files pass per-file checks
  Cross-checking coverage against raw .mat files...
  ✅ Coverage OK — all expected conditions found in output

  sub-08 [RCS]
  Found 30 .npz files
  ✅ All 30 files pass per-file checks
  Cross-checking coverage against raw .mat files...
⚠️ Length mismatch detected. Truncating to 371842 samples.
⚠️ Length mismatch detected. Truncating to 167161 samples.
⚠️ Length mismatch detected. Truncating to 604888 samples.
⚠️ Length mismatch de